# ContractRisk — diagnóstico visual 18

La prueba 17 terminó técnicamente, pero el OCR añadió un cero a un importe y el control bloqueó la respuesta. Este diagnóstico **post-hoc** tiene ocho imágenes ya congeladas en el repositorio: no cambia modelos, prompts ni guardas. No sustituye resultados anteriores ni constituye validación ciega.

Una sola pasada en **GPU T4**, conservando todas las salidas. La captura original se replica una vez para estudiar OCR; la pregunta de esta tanda solicita modalidad y valor, no la entidad de la corrida 17. Otros casos comparan formatos de dos fichas e incluyen tres controles. No hay API del juez ni gastos adicionales. No modifica main ni el Space.

Ejecutar en orden. Si falla preparación, ir directamente a la última celda para conservar evidencia; no ejecutar inferencia ni repetir por cuenta propia.

In [ ]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['ANONYMIZED_TELEMETRY']='false'
SETUP_OK=False
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_visual18_',dir='/content'))/'repo'
subprocess.run(['git','clone','--single-branch','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido. Conservar diagnóstico con la última celda: '+logfile)


In [ ]:
SETUP_OK=False
run_visible([sys.executable,'final/prepare_test_history.py'],'visual18_history.log')
run_visible([sys.executable,'-m','pip','install','-r','final/space_candidate/requirements.txt'],'visual18_install.log')
run_visible([sys.executable,'final/service_environment.py','--output','visual18_environment.json'],'visual18_environment.log')
run_visible([sys.executable,'-c',"import torch; assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Selecciona T4'; print(torch.cuda.get_device_name(0))"],'visual18_gpu.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-v'],'visual18_tests_final.log')
run_visible([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],'visual18_tests_m3.log')
from final.visual_deployment18 import load_protocol
protocol=load_protocol()
print('Casos congelados:',[c['id'] for c in protocol['cases']])
SETUP_OK=True


In [ ]:
if not SETUP_OK:raise RuntimeError('Preparación incompleta. Descargar diagnóstico sin ejecutar inferencia.')
RESULTS=ROOT/'visual18_results'
run_visible([sys.executable,'-u','final/visual_deployment18.py','run','--output',str(RESULTS)],'visual18_inference.log')


## Lectura y evidencia

`completed_requires_review` significa que terminó la tanda, no que se aprobaron las respuestas o el despliegue. Los cinco casos legibles y tres controles se leen por separado. Una abstención por OCR errado no cuenta como lectura correcta. Los chequeos de importes/modalidad son literales, no revisión semántica humana.

Se conservan transcripciones, respuestas, fuentes, guardas y resultados parciales. El ZIP y el notebook con salidas permiten la revisión posterior. No publicar archivos en el Space ni repetir para buscar un mejor resultado.

In [ ]:
import zipfile
from google.colab import files
metadata=ROOT/'visual18_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'La inferencia no comenzó o no generó metadata; consultar logs.')
archive=ROOT.parent/'contractrisk_final_visual18_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['visual18_results','final/evidence/visual18_protocol']:
        base=ROOT/folder
        if base.exists():
            for item in sorted(base.rglob('*')):
                if item.is_file():z.write(item,item.relative_to(ROOT))
    for pattern in ['visual18_*.log','visual18_environment.json']:
        for item in sorted(ROOT.glob(pattern)):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
    z.writestr('setup_state.json',json.dumps({'setup_ok':SETUP_OK}))
files.download(str(archive))
